# Controllo PER su FlyWire v783

Notebook gia configurato. Non modificare la cella config.

1. Runtime -> Run all
2. Aspetta (su Colab il run zucchero puo richiedere 10-20 minuti)
3. In fondo compaiono i rate di MN9 sinistro e destro


In [ ]:
# Solo Colab: scarica il modello Shiu e Brian2
import os
if not os.path.exists("Drosophila_brain_model/model.py"):
    !git clone --depth 1 https://github.com/philshiu/Drosophila_brain_model.git
    !pip install -q brian2 pandas pyarrow
%cd Drosophila_brain_model
print("cwd:", os.getcwd())
print("file 783 presenti:",
      os.path.exists("Completeness_783.csv"),
      os.path.exists("Connectivity_783.parquet"))


In [ ]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

# PUNTO 2 gia fatto: snapshot pubblico FlyWire v783 (stessi ID di Codex)
config = {
    "path_res": "./results/per_control",
    "path_comp": "./Completeness_783.csv",
    "path_con": "./Connectivity_783.parquet",
    "n_proc": -1,
}

# 3 trial invece di 30: piu veloce per il controllo. Il paper usa 30.
params["n_run"] = 3
params["r_poi"] = 100 * Hz


In [ ]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]

MN9_L = 720575940618238523
MN9_R = 720575940660219265


In [ ]:
run_exp(exp_name="sugar_100Hz", neu_exc=neu_sugar, params=params, **config)


In [ ]:
df_spike = utl.load_exps(["./results/per_control/sugar_100Hz.parquet"])
df_rate, df_rate_std = utl.get_rate(
    df_spike, t_run=params["t_run"], n_run=params["n_run"]
)

print("=== INCOLLA QUESTI NUMERI ===")
for name, nid in [("MN9_L", MN9_L), ("MN9_R", MN9_R)]:
    if nid in df_rate.index:
        mu = df_rate.loc[nid].iloc[0]
        sd = df_rate_std.loc[nid].iloc[0] if nid in df_rate_std.index else float("nan")
        print(f"{name} {nid}: {mu:.2f} +/- {sd:.2f} Hz")
    else:
        print(f"{name} {nid}: NESSUNO SPIKE (0 Hz o ID assente nel grafo 783)")
